# Scraping de descuentos por categoría en Alkosto

Extrae los **descuentos más grandes por categoría** desde la página de ofertas:
**https://www.alkosto.com/ofertas/c/ofertas**

Usa la **API de Algolia** directamente (rápido, sin navegador).

In [1]:
import requests
import pandas as pd
import time
import random
from typing import List, Dict
from urllib.parse import urlencode

## Configuración Algolia (verificada)

In [2]:
ALGOLIA_APP_ID = "QX5IPS1B1Q"
ALGOLIA_API_KEY = "7a8800d62203ee3a9ff1cdf74f99b268"
ALGOLIA_INDEX = "alkostoIndexAlgoliaPRD"
ALGOLIA_URL = f"https://{ALGOLIA_APP_ID.lower()}-dsn.algolia.net/1/indexes/{ALGOLIA_INDEX}/query"

# Mapeo categorías Alkosto → Negocio
CATEGORIA_MAP = {
    "Celulares": "Tecnología", "Smartphones": "Tecnología",
    "Computadores y Tablet": "Tecnología", "Computadores Portátiles": "Tecnología",
    "Portátiles Laptops y Convertibles 2 en 1": "Tecnología", "Tabletas": "Tecnología",
    "TV": "Tecnología", "Smart TV": "Tecnología", "Televisores Samsung": "Tecnología",
    "Televisores LG": "Tecnología", "Televisores Kalley": "Tecnología",
    "Monitores": "Tecnología", "Consolas y Videojuegos": "Tecnología",
    "Electrodomésticos": "Hogar", "Refrigeración": "Hogar", "Lavado": "Hogar",
    "Neveras": "Hogar", "Lavadoras": "Hogar", "Audio Para el Hogar": "Hogar",
    "Baño": "Hogar", "Cocina": "Hogar", "Artículos Cocina": "Hogar",
    "Aires Acondicionados": "Hogar", "Aspiradoras": "Hogar",
    "Camas y Bases Camas": "Hogar", "Armarios": "Hogar", "Muebles": "Hogar",
    "Hogar": "Hogar",
    "Accesorios Automotores": "Movilidad", "Accesorios Moto": "Movilidad",
    "Bicicletas Spinning y Estáticas": "Movilidad", "Bicicletas": "Movilidad",
    "Accesorios Carro": "Movilidad", "Baterías de Carro": "Movilidad",
    "Audio": "Entretenimiento", "Audífonos": "Entretenimiento",
    "Audifonos y Parlantes Gamers": "Entretenimiento", "Barras de Sonido": "Entretenimiento",
    "Cámaras": "Entretenimiento", "Cámaras Fotográficas": "Entretenimiento",
    "Juguetes": "Entretenimiento", "Consolas": "Entretenimiento",
    "Accesorios de Electrónica": "Accesorios", "Accesorios Computadores": "Accesorios",
    "Accesorios Celulares y Tabletas": "Accesorios", "Cables, Cargadores, Adaptadores": "Accesorios",
    "Carcasas, Estuches Y Protectores": "Accesorios", "Accesorios Audio": "Accesorios",
    "Accesorios Gaming": "Accesorios", "Baterías Externas PowerBank": "Accesorios",
    "Relojes y Anillos inteligentes": "Accesorios", "Anillos Inteligentes": "Accesorios",
    "Accesorios Tv y Video": "Accesorios", "Accesorios Videojuegos": "Accesorios",
    "Accesorios Cámaras": "Accesorios", "Accesorios Nintendo": "Accesorios",
    "Accesorios PlayStation": "Accesorios",
}

# Categorías a consultar
ALKOSTO_CATEGORIES = [
    "Electrodomésticos", "TV", "Smart TV", "Computadores y Tablet",
    "Refrigeración", "Lavado", "Computadores Portátiles",
    "Portátiles Laptops y Convertibles 2 en 1", "Lavadoras",
    "Celulares", "Smartphones", "Neveras",
    "Audio Para el Hogar", "Accesorios de Electrónica",
    "Accesorios Computadores", "Accesorios Celulares y Tabletas",
    "Consolas y Videojuegos", "Audio", "Audífonos",
    "Relojes y Anillos inteligentes", "Monitores"
]

In [3]:
def fetch_category(category: str, limit: int = 200) -> List[Dict]:
    """Obtiene productos con descuento de una categoría."""
    products = []
    page = 0
    hits_per_page = 50
    
    while len(products) < limit:
        filter_str = f"categoryname_text_es_mv:'{category}' AND discountprice_double>0"
        payload = {
            "params": urlencode({
                "hitsPerPage": hits_per_page,
                "page": page,
                "filters": filter_str,
                "attributesToRetrieve": "objectID,code_string,name_text_es,brand_string_mv,categoryname_text_es_mv,baseprice_cop_string,discountprice_double",
                "attributesToHighlight": ""
            })
        }
        headers = {
            "X-Algolia-API-Key": ALGOLIA_API_KEY,
            "X-Algolia-Application-Id": ALGOLIA_APP_ID,
            "Content-Type": "application/json"
        }
        try:
            resp = requests.post(ALGOLIA_URL, json=payload, headers=headers, timeout=30)
            resp.raise_for_status()
            result = resp.json()
        except Exception as e:
            print(f"  Error en {category} pagina {page}: {e}")
            break
        
        hits = result.get('hits', [])
        if not hits:
            break
        
        for h in hits:
            base = h.get('baseprice_cop_string')
            disc = h.get('discountprice_double')
            if isinstance(base, str):
                try: base = float(base)
                except: base = None
            if base and disc and base > 0:
                pct = round((1 - disc / base) * 100)
                if pct > 0:
                    name = h.get('name_text_es') or h.get('_highlightResult', {}).get('name_text_es', {}).get('value', 'N/A')
                    brand = h.get('brand_string_mv', ['N/A'])[0] if h.get('brand_string_mv') else 'N/A'
                    products.append({
                        'categoria_alkosto': category,
                        'categoria_negocio': CATEGORIA_MAP.get(category, 'Otra'),
                        'marca': brand,
                        'nombre': name[:80],
                        'precio_original': base,
                        'precio_descuento': disc,
                        'descuento_pct': pct,
                    })
        
        if len(hits) < hits_per_page:
            break
        page += 1
        time.sleep(random.uniform(0.3, 0.6))
    
    return products

## Ejecutar: Top N descuentos por categoría

In [4]:
# Configuración
TOP_N = 5
LIMIT_PER_CAT = 200

all_results = []

print(f"Consultando {len(ALKOSTO_CATEGORIES)} categorias...")
print(f"Top {TOP_N} por categoria, revisando hasta {LIMIT_PER_CAT} productos cada una\n")

for cat in ALKOSTO_CATEGORIES:
    print(f"🔍 {cat}...")
    products = fetch_category(cat, limit=LIMIT_PER_CAT)
    
    if not products:
        print(f"   (sin descuentos)")
        continue
    
    products.sort(key=lambda x: x['descuento_pct'], reverse=True)
    top = products[:TOP_N]
    
    for i, p in enumerate(top, 1):
        desc_str = f"   {i}. [{p['descuento_pct']}%] {p['marca']} - {p['nombre']}"
        price_str = f"       ${p['precio_original']:,.0f} → ${p['precio_descuento']:,.0f}"
        print(desc_str)
        print(price_str)
    
    all_results.extend(top)
    time.sleep(0.2)

print(f"\n✅ Total productos mostrados: {len(all_results)}")

Consultando 21 categorias...
Top 5 por categoria, revisando hasta 200 productos cada una

🔍 Electrodomésticos...
   1. [63%] KALLEY - Freidora de Aire KALLEY 4.5 Litros K-MAF45 Negro
       $539,900 → $199,900
   2. [60%] KALLEY - Freidora de Aire KALLEY 3.5Litros K-MAF35 Negro
       $459,900 → $184,900
   3. [58%] KALLEY - Freidora de Aire KALLEY  6.3 Litros K-MAF6 Negro
       $639,900 → $269,900
   4. [57%] KALLEY - Nevecón KALLEY Side by Side 529 Litros K-N529L2 Gris
       $6,299,900 → $2,699,900
   5. [56%] KALLEY - Nevecón KALLEY Tipo Europeo 362 Litros K-N362L4 Gris
       $5,699,900 → $2,519,900
🔍 TV...
   1. [61%] SAMSUNG - TV SAMSUNG 75" Pulgadas 195,6 cm 75R85H 4K-UHD Mini LED-MicroRGB Smart TV con IA
       $19,169,900 → $7,499,900
   2. [60%] KALLEY - TV KALLEY 50" Pulgadas 126 cm 50G315 4K-UHD MAX LED Smart TV Google
       $3,099,900 → $1,249,900
   3. [60%] KALLEY - TV KALLEY 50" Pulgadas 126 cm 50G315A 4K-UHD MAX LED Smart TV Google
       $3,099,900 → $1,249,900
   

## Resumen por categoría de negocio

In [5]:
if all_results:
    df = pd.DataFrame(all_results)
    
    print("=== TOP DESCUENTOS POR CATEGORIA DE NEGOCIO ===\n")
    
    for cat_negocio in sorted(df['categoria_negocio'].unique()):
        cat_df = df[df['categoria_negocio'] == cat_negocio].copy()
        cat_df = cat_df.sort_values('descuento_pct', ascending=False)
        
        print(f"📦 {cat_negocio} ({len(cat_df)} categorias Alkosto)")
        print(f"   Descuento promedio: {cat_df['descuento_pct'].mean():.1f}% | Max: {cat_df['descuento_pct'].max()}%")
        
        for _, row in cat_df.head(TOP_N).iterrows():
            p_str = f"   • [{row['descuento_pct']}%] {row['categoria_alkosto']} | {row['marca']} | {row['nombre'][:55]}"
            pr_str = f"       ${row['precio_original']:,.0f} → ${row['precio_descuento']:,.0f}"
            print(p_str)
            print(pr_str)
        print()
    
    print("=== TOP 10 ABSOLUTOS (todas las categorias) ===")
    top10 = df.nlargest(10, 'descuento_pct')
    for rank, (_, row) in enumerate(top10.iterrows(), 1):
        line = f"{rank}. [{row['descuento_pct']}%] {row['categoria_negocio']}/{row['categoria_alkosto']} | {row['marca']} | {row['nombre'][:50]}"
        pr_str = f"    ${row['precio_original']:,.0f} → ${row['precio_descuento']:,.0f}"
        print(line)
        print(pr_str)

=== TOP DESCUENTOS POR CATEGORIA DE NEGOCIO ===

📦 Accesorios (20 categorias Alkosto)
   Descuento promedio: 66.2% | Max: 85%
   • [85%] Accesorios de Electrónica | HYPERX | Audífonos de Diadema HYPERX Inalámbricos Over Ear Gamin
       $389,900 → $59,900
   • [85%] Accesorios Computadores | HYPERX | Audífonos de Diadema HYPERX Inalámbricos Over Ear Gamin
       $389,900 → $59,900
   • [75%] Accesorios Celulares y Tabletas | BACKBONE | Estuche BACKBONE Blanco Edición PLAYSTATION para Backbo
       $119,900 → $29,900
   • [72%] Accesorios de Electrónica | KALLEY | Cable KALLEY USB-C a USB-C de 1.0 Metro Negro
       $59,900 → $16,900
   • [72%] Accesorios de Electrónica | KALLEY | Cable KALLEY USB a USB-C de 1.0 Metro Negro
       $59,900 → $16,900

📦 Entretenimiento (10 categorias Alkosto)
   Descuento promedio: 68.6% | Max: 71%
   • [71%] Audio | LG | Barra de Sonido LG S20A 50 Watts Negro
       $799,900 → $229,900
   • [70%] Audio | PANASONIC | Audífonos PANASONIC Inalámbricos Bluet

In [6]:
# ── Fin del notebook ──